# glass5: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('21_glass5_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.0,100.000,50.000,79.167,97.073,0.000
smote,67.935,61.333,66.667,60.0,99.024,79.512,87.500,98.049,64.634
bsmote,67.935,61.333,66.667,60.0,99.024,79.512,87.500,98.049,64.634
ros,67.812,58.000,61.667,60.0,98.780,79.390,85.714,97.561,62.906
adasyn,67.935,61.333,66.667,60.0,99.024,79.512,87.857,98.293,64.634
casb,67.935,61.333,66.667,60.0,99.024,79.512,86.345,97.805,64.634
casb_relaxed,67.935,61.333,66.667,60.0,99.024,79.512,87.179,98.049,64.634
casb_edited,67.441,54.667,56.667,60.0,98.049,79.024,88.095,98.415,61.054
casb_ros,81.954,72.667,83.333,70.0,98.780,84.390,85.952,97.683,77.311


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,54.142,53.333,60.000,50.0,100.000,75.000,90.465,94.390,53.738
smote,95.567,81.556,79.048,95.0,97.073,96.037,86.583,98.841,88.561
bsmote,98.371,83.556,77.381,100.0,96.829,98.415,86.583,98.841,90.963
ros,86.903,67.667,66.667,80.0,96.585,88.293,83.798,94.146,77.285
adasyn,92.266,71.889,70.714,90.0,96.341,93.171,86.583,98.841,82.078
casb,90.200,79.667,80.000,85.0,97.317,91.159,84.161,96.220,84.933
casb_relaxed,89.830,73.000,70.000,85.0,96.585,90.793,85.774,98.476,81.415
casb_edited,88.953,63.333,58.333,85.0,94.878,89.939,79.107,98.232,76.143
casb_ros,86.903,67.667,66.667,80.0,96.585,88.293,83.798,94.146,77.285


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,34.142,33.333,40.000,30.0,100.000,65.000,100.000,100.000,33.738
smote,79.877,78.000,76.667,80.0,99.756,89.878,100.000,100.000,78.939
bsmote,74.019,71.333,76.667,70.0,99.756,84.878,97.000,99.634,72.676
ros,83.933,79.667,81.667,80.0,99.512,89.756,94.167,99.634,81.800
adasyn,84.019,81.333,86.667,80.0,99.756,89.878,98.333,99.878,82.676
casb,84.019,81.333,86.667,80.0,99.756,89.878,100.000,100.000,82.676
casb_relaxed,79.877,78.000,76.667,80.0,99.756,89.878,100.000,100.000,78.939
casb_edited,76.948,74.667,76.667,75.0,99.756,87.378,100.000,100.000,75.808
casb_ros,93.772,88.000,91.667,90.0,99.268,94.634,98.333,99.878,90.886


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,20.000,20.000,20.000,20.0,100.000,60.000,76.190,97.805,20.000
smote,95.110,68.524,55.333,95.0,95.854,95.427,93.333,99.024,81.817
bsmote,95.110,68.524,55.333,95.0,95.854,95.427,93.333,99.024,81.817
ros,98.015,72.667,60.000,100.0,96.098,98.049,93.333,99.024,85.341
adasyn,98.386,78.667,68.333,100.0,96.829,98.415,93.333,99.024,88.526
casb,95.231,70.905,59.333,95.0,96.098,95.549,93.333,99.024,83.068
casb_relaxed,98.015,72.667,60.000,100.0,96.098,98.049,93.333,99.024,85.341
casb_edited,97.889,74.381,62.333,100.0,95.854,97.927,93.333,99.024,86.135
casb_ros,97.886,72.333,60.000,100.0,95.854,97.927,93.333,99.024,85.109


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,27.071,26.667,30.000,25.00,100.000,62.500,86.456,97.317,26.869
smote,84.622,72.353,69.429,82.50,97.927,90.213,91.854,98.979,78.488
bsmote,83.859,71.187,69.012,81.25,97.866,89.558,91.104,98.887,77.523
ros,84.166,69.500,67.500,80.00,97.744,88.872,89.253,97.591,76.833
adasyn,85.652,73.306,73.095,82.50,97.988,90.244,91.527,99.009,79.479
casb,84.346,73.310,73.167,80.00,98.049,89.024,90.960,98.262,78.828
casb_relaxed,83.914,71.250,68.333,81.25,97.866,89.558,91.571,98.887,77.582
casb_edited,82.808,66.762,63.500,80.00,97.134,88.567,90.134,98.918,74.785
casb_ros,90.129,75.167,75.417,85.00,97.622,91.311,90.354,97.683,82.648


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,81.954,smote,67.935,14.019,WIN
1,f1,casb_ros,72.667,smote,61.333,11.333,WIN
2,precision,casb_ros,83.333,smote,66.667,16.667,WIN
3,recall,casb_ros,70.000,smote,60.000,10.000,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,90.200,bsmote,98.371,-8.171,LOSS
1,f1,casb,79.667,bsmote,83.556,-3.889,LOSS
2,precision,casb,80.000,smote,79.048,0.952,WIN
3,recall,casb,85.000,bsmote,100.000,-15.000,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,93.772,adasyn,84.019,9.753,WIN
1,f1,casb_ros,88.000,adasyn,81.333,6.667,WIN
2,precision,casb_ros,91.667,adasyn,86.667,5.000,WIN
3,recall,casb_ros,90.000,smote,80.000,10.000,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,90.129,adasyn,85.652,4.477,WIN
1,f1,casb_ros,75.167,adasyn,73.306,1.861,WIN
2,precision,casb_ros,75.417,adasyn,73.095,2.321,WIN
3,recall,casb_ros,85.000,smote,82.500,2.500,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.0,1.000000,0.500000,0.625000,0.926829,0.000000,None
1,42,1,svm,smote,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,0.625000,0.926829,0.686887,None
2,42,1,svm,bsmote,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,0.625000,0.926829,0.686887,None
3,42,1,svm,ros,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,0.642857,0.939024,0.686887,None
4,42,1,svm,adasyn,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,0.642857,0.939024,0.686887,None
5,42,1,svm,casb,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,0.642857,0.939024,0.686887,None
6,42,1,svm,casb_relaxed,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,0.642857,0.939024,0.686887,None
7,42,1,svm,casb_edited,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,0.666667,0.951220,0.686887,None
8,42,1,svm,casb_ros,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,0.642857,0.939024,0.686887,None
9,42,1,knn,original,0.50,0.707107,0.666667,1.000000,0.5,1.000000,0.750000,1.000000,1.000000,0.686887,None
